# Ranked Bars, Annotation and Encoding

**DS4DH Practice Pack · Module 10 — Data Visualization and Communication**

*Technique:* Encoding choices as argument — what a chart claims before anyone reads it

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/ds4dh-colab-pack/blob/main/notebooks/10b_ranked_bars.ipynb)

Data: `merged_dataset.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# This notebook reads the CSVs sitting next to it. In Colab you will be asked
# to upload them from the pack's data/ folder.
NEEDED = ['merged_dataset.csv']

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('merged_dataset.csv')
CITIES = ['Montréal', 'Toronto', 'Edmonton', 'Vancouver']

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Every chart makes an argument. Sort order, axis limits, colour and reference lines
are not decoration — each is a claim about what matters, made before the reader
has consciously processed anything.

This notebook builds the same chart four ways, each honest, each saying something
different.

In [ ]:
# One row per Census Subdivision.
#   • rows with no csd_code are CMA-level and Canada-level aggregates, not CSDs
#   • each CSD appears 3x (Immigrant / Non-immigrants / Total Immigrant Status)
# Keeping either would silently double- or triple-count places.
csd = df.dropna(subset=['csd_code'])
base = csd[(csd['immigrant_status'] == 'Total Immigrant Status')
           & (csd['cma'].isin(CITIES))].copy()

print(f'{len(df):>4} rows in the raw file')
print(f'{len(csd):>4} after dropping CMA/Canada aggregate rows')
print(f'{len(base):>4} CSDs in the four cities (one row each)')

In [ ]:
d = base.dropna(subset=['Total', 'tot_pop'])
show = d.nlargest(12, 'Total').sort_values('Total')
labels = [n[:28] for n in show['geography_name']]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(labels, show['Total'], color='#3DA5D9')
ax.set_xlabel('Total STIR (%)')
ax.set_title('Version 1 — the default')
plt.tight_layout()
plt.show()

Version 1 is not wrong. It is also not saying anything: no reference point, no
indication of which places are large, no explanation of why 12.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#E8663D' if v > 30 else '#3DA5D9' for v in show['Total']]
ax.barh(labels, show['Total'], color=colors)
ax.axvline(30, color='#333', ls='--', lw=1.5)
ax.text(30.3, -0.6, '30% affordability threshold', fontsize=9, color='#333')
ax.set_xlabel('Total STIR (%)')
ax.set_title('Version 2 — with the threshold that makes the numbers mean something')
plt.tight_layout()
plt.show()

print(f'{(show["Total"] > 30).sum()} of these {len(show)} CSDs are above the threshold.')

Version 2 adds the one piece of context a housing reader needs. The colour now
encodes a policy category rather than being arbitrary — the chart says *these
places are over the line*, which is a claim the first version left the reader to
work out.

In [ ]:
# Version 3 — bar length is burden, bar colour is population. Two variables,
# one chart, without a second axis.
fig, ax = plt.subplots(figsize=(10, 6))
norm = plt.Normalize(np.log10(show['tot_pop']).min(), np.log10(show['tot_pop']).max())
cmap = plt.get_cmap('viridis')
bars = ax.barh(labels, show['Total'],
               color=[cmap(norm(v)) for v in np.log10(show['tot_pop'])])
ax.axvline(30, color='#333', ls='--', lw=1.5)
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cb = fig.colorbar(sm, ax=ax)
cb.set_label('log10 population')
ax.set_xlabel('Total STIR (%)')
ax.set_title('Version 3 — colour warns which bars rest on few households')
plt.tight_layout()
plt.show()

In [ ]:
# Version 4 — the honest chart: filter to places big enough to be stable,
# and say so in the title.
MIN_POP = 10000
big = d[d['tot_pop'] >= MIN_POP].nlargest(12, 'Total').sort_values('Total')

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh([n[:28] for n in big['geography_name']], big['Total'],
        color=['#E8663D' if v > 30 else '#3DA5D9' for v in big['Total']])
ax.axvline(30, color='#333', ls='--', lw=1.5)
ax.set_xlabel('Total STIR (%)')
ax.set_title(f'Highest housing burden, CSDs with population ≥ {MIN_POP:,}')
plt.tight_layout()
plt.show()

overlap = set(big['csd_code']) & set(show['csd_code'])
print(f'{len(overlap)} of the original 12 survive the population filter.')

### 🔧 Your turn 1

Change `MIN_POP` to 50000 and re-run version 4.

How much of the chart changes? If the "worst places" list is largely different,
which chart would you put in a public document — and does the answer depend on
who the audience is?

## The axis-limit question

Starting a bar chart's axis anywhere but zero exaggerates differences, because bar
length is the encoding and a truncated bar no longer encodes the value.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, lo, title in [(axes[0], 0, 'zero baseline — honest for bars'),
                      (axes[1], 25, 'truncated at 25 — differences look 4x larger')]:
    ax.barh([n[:22] for n in big['geography_name']][-6:], big['Total'][-6:],
            color='#3DA5D9')
    ax.set_xlim(lo, big['Total'].max() + 2)
    ax.set_title(title)
    ax.set_xlabel('Total STIR (%)')
plt.tight_layout()
plt.show()

rng_full = big['Total'][-6:].max() - big['Total'][-6:].min()
print(f'The actual spread among these six is {rng_full:.1f} percentage points.')
print('The right-hand chart makes it look like the difference between them is')
print('most of the story. It is not.')

### 🔧 Your turn 2

Truncating is legitimate for line charts and dot plots, where position rather
than length is the encoding.

Redraw the truncated version as a dot plot (`ax.scatter`) and decide whether the
same truncation still misleads. What does that tell you about the rule?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** At `MIN_POP = 50000` most of the list changes again. For a public
document use the filtered chart, and say what the filter was — but the answer does
depend on audience: a municipal affairs reader may genuinely want small
municipalities included, because those are governments with their own housing
powers. The rule is not "always filter"; it is "make the filter visible so the
reader can tell what population the chart describes".

**Your turn 2.** Truncation is much less misleading on a dot plot, because
position encodes the value and no visual quantity is being cut in half. The rule
is therefore not about axes at all — it is about *encoding*. Bars encode with
length, so length must start at zero. Points encode with position, so the axis can
start where the data is. Stating the rule as "never truncate an axis" gets the
right answer for the wrong reason and then misapplies it.

</details>

## Where this stops

One chart, made deliberately. The next notebook is about several charts that have
to be read against each other.